# Create folder structure


In [0]:
from pyspark.sql.functions import *

landing_path = "/Volumes/cat_medallion/source_1/vol_poc1/poc2/autoloader/landing/customers"

# Drop the folder and its contents
dbutils.fs.rm(landing_path, True)

# Recreate the folder
dbutils.fs.mkdirs(landing_path)

print("Customers landing folder recreated.")

dbutils.fs.ls('/Volumes/cat_medallion/source_1/vol_poc1/poc2/autoloader/landing/customers')

# variables

In [0]:
BASE = "/Volumes/cat_medallion/source_1/vol_poc1/poc2/autoloader"

CUSTOMERS_LANDING     = f"{BASE}/landing/customers"
CUSTOMERS_SCHEMA      = f"{BASE}/schema/customers"
CUSTOMERS_CHECKPOINT  = f"{BASE}/checkpoints/customers"

BRONZE_CUSTOMERS = "cat_medallion.source_1.poc2_bronze_customers"

# Start Streaming Read

In [0]:
df_read_cust=spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","csv") \
    .option("cloudFiles.schemaLocation",CUSTOMERS_SCHEMA) \
    .option("cloudFiles.InferColumnTypes",True) \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .option("header",True) \
    .load(CUSTOMERS_LANDING)

# Start Streaming Write

In [0]:
from pyspark.sql.functions import col, current_timestamp

df_bronze_cust = (
    df_read_cust
    .withColumn("_ingested_at", current_timestamp())
    .withColumn(
        "_source_file",
        col("_metadata.file_path")
    )
)

(
    df_bronze_cust.writeStream
    .format("delta")
    .option("checkpointLocation", CUSTOMERS_CHECKPOINT)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(BRONZE_CUSTOMERS)
)

In [0]:
print("Is active:", df_bronze_cust.isActive)
print("Last progress:")
print(df_bronze_cust.lastProgress)

spark.sql(f"SELECT COUNT(*) AS cnt FROM {BRONZE_CUSTOMERS}").show()

spark.sql(f"""
    SELECT *
    FROM {BRONZE_CUSTOMERS}
    ORDER BY customer_id
""").show()

dbutils.fs.ls(CUSTOMERS_CHECKPOINT)

# Generic function

In [0]:
from pyspark.sql import functions as F

def ingest_to_bronze(landing_path, schema_path, checkpoint_path, target_table, file_format="csv"):
    reader = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", file_format)
            .option("cloudFiles.schemaLocation", schema_path)
            .option("cloudFiles.inferColumnTypes", "true")
            .option("header", "true")
    )

    stream_df = (
        reader.load(landing_path)
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    query = (
        stream_df.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_path)
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(target_table)
    )
    return query

# generic validation query

In [0]:
for tbl in [BRONZE_CUSTOMERS]:
    spark.sql(f"""
        SELECT
            _source_file,
            COUNT(*)                AS row_count,
            MIN(_ingested_at)       AS first_ingested,
            MAX(_ingested_at)       AS last_ingested
        FROM {tbl}
        GROUP BY _source_file
        ORDER BY _source_file
    """).show(truncate=False)

    spark.sql(f"SELECT COUNT(*) AS total_rows FROM {tbl}").show()

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT _source_file) AS distinct_files,
           MIN(_ingested_at) AS first_ingest,
           MAX(_ingested_at) AS last_ingest
    FROM {BRONZE_CUSTOMERS}
""").show(truncate=False)